# Nguyễn Thành Trí - MSSV: 6451071079
## Bài 2 (Trang 28-29) - Câu 5: Tiền xử lý dữ liệu thiếu và xử lý giá trị 'unknown'

### Đề bài:
Tiền xử lý các thuộc tính thiếu dữ liệu / giá trị `'unknown'` và loại bỏ các thuộc tính không cần thiết:
1. Xử lý giá trị `'unknown'` ở các cột có tỷ lệ nhỏ (`job`, `education`) bằng giá trị xuất hiện nhiều nhất (mode).
2. Xử lý thuộc tính `contact`: thay thế 'unknown' bằng giá trị phổ biến nhất hoặc mã hóa.
3. Tạo thuộc tính nhị phân đại diện cho việc khách hàng đã từng được liên hệ trong chiến dịch trước hay chưa (`is_previous_contact`) dựa vào `pdays != -1`.
4. Loại bỏ các thuộc tính không còn cần thiết hoặc chứa quá nhiều giá trị thiếu nghiêm trọng như `poutcome`.

In [1]:
# Nguyễn Thành Trí - 6451071079
# Bài 2 (Trang 28-29) - Câu 5: Tiền xử lý dữ liệu thiếu và làm sạch

import os
import pandas as pd
import numpy as np

print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

def get_data_path(rel_path):
    candidates = [
        rel_path,
        os.path.join("..", rel_path),
        os.path.join("..", "..", rel_path),
        os.path.join("datasets", os.path.basename(rel_path)),
        os.path.join("..", "datasets", os.path.basename(rel_path)),
        os.path.join("..", "..", "datasets", os.path.basename(rel_path)),
        os.path.join("..", "..", "datasets", "Bank", os.path.basename(rel_path)),
        os.path.join("datasets", "Bank", os.path.basename(rel_path))
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Không tìm thấy file dữ liệu: {rel_path}")

file_path = get_data_path("bank.csv")
df = pd.read_csv(file_path, sep=';')
print(f"Kích thước ban đầu: {df.shape}")

# 1. Điền giá trị mode cho 'job' (38 unknown) và 'education' (187 unknown)
job_mode = df[df['job'] != 'unknown']['job'].mode()[0]
edu_mode = df[df['education'] != 'unknown']['education'].mode()[0]
contact_mode = df[df['contact'] != 'unknown']['contact'].mode()[0]

print(f"Giá trị mode thay thế: job='{job_mode}', education='{edu_mode}', contact='{contact_mode}'")

df['job'] = df['job'].replace('unknown', job_mode)
df['education'] = df['education'].replace('unknown', edu_mode)
df['contact'] = df['contact'].replace('unknown', contact_mode)

# 2. Tạo thuộc tính mới 'is_previous_contact': 1 nếu pdays != -1, ngược lại 0
df['is_previous_contact'] = (df['pdays'] != -1).astype(int)

# 3. Loại bỏ thuộc tính 'poutcome' do thiếu đến 81.95% và cột 'pdays' đã được trích xuất
df = df.drop(columns=['poutcome', 'pdays'])

print()
print(f"Kích thước sau khi làm sạch: {df.shape}")
print("Kiểm tra lại số lượng giá trị 'unknown' còn lại trên mỗi cột:")
for col in df.select_dtypes(include='object').columns:
    c = (df[col].astype(str).str.lower() == 'unknown').sum()
    print(f"  {col}: {c}")

display(df.head())

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079
Kích thước ban đầu: (4521, 17)
Giá trị mode thay thế: job='management', education='secondary', contact='cellular'

Kích thước sau khi làm sạch: (4521, 16)
Kiểm tra lại số lượng giá trị 'unknown' còn lại trên mỗi cột:
  job: 0
  marital: 0
  education: 0
  default: 0
  housing: 0
  loan: 0
  contact: 0
  month: 0
  y: 0


/tmp/ipykernel_217535/3453140471.py:50: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,previous,y,is_previous_contact
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,0,no,0
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,4,no,1
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,1,no,1
3,30,management,married,tertiary,no,1476,yes,yes,cellular,3,jun,199,4,0,no,0
4,59,blue-collar,married,secondary,no,0,yes,no,cellular,5,may,226,1,0,no,0


### Nhận xét & Đánh giá (Câu 5):
1. **Khắc phục giá trị thiếu**:
   - `job` được điền bằng mode `'management'` (38 dòng).
   - `education` được điền bằng mode `'secondary'` (187 dòng).
   - `contact` được điền bằng mode `'cellular'` (1,324 dòng).
   - Sau bước này, tập dữ liệu không còn bất kỳ giá trị `'unknown'` nào trong các thuộc tính này.
2. **Trích xuất thông tin hữu ích**: Cột `pdays` (-1 đại diện cho chưa từng liên hệ) đã được chuyển đổi thành biến chỉ báo nhị phân `is_previous_contact` (1: Đã từng liên hệ, 0: Chưa từng liên hệ), giúp mô hình học dễ dàng mà không bị ảnh hưởng bởi giá trị âm `-1`.
3. **Loại bỏ thuộc tính nhiễu**: Cột `poutcome` chứa 82% giá trị không xác định được loại bỏ an toàn nhằm tránh làm sai lệch mô hình phân tích.